# Projeto 2: Como pegar dados em sites e automatizar a criação da carteira do nosso modelo.

### Desafio:

* Construir um código que vá no fundamentus.com e busque dados de todos os indicadores da bolsa brasileira e gere a carteira selecionando as maiores ev_ebit e roic.    


### Passo a passo:

   **Passo 1** - Definir um navegador que você irá utilizar para navegar com o Python.

   **Passo 2** - Importar os módulos e bibliotecas.
   
   **Passo 3** - Entender como funcionam requisições na internet.
   
   **Passo 4** - Entender como sites funcionam.
   
   **Passo 5** - Conhecer e mapear o processo de coleta de dados no site do Fundamentus. 
   
   **Passo 6** - Ler a tabela de dados.
   
   **Passo 7** - Construir a tabela final.

# Passo 1: Escolher o navegador.

No nosso caso, utilizaremos o Google Chrome. 

# Passo 2: Importar as bibliotecas.

In [ ]:
!pip install selenium

In [2]:
!pip install webdriver-manager

  Obtaining dependency information for webdriver-manager from https://files.pythonhosted.org/packages/19/5a/a16653bfce685c9832217d377f52065351eeac9862e44e2996cd81f3bb4d/webdriver_manager-4.0.0-py2.py3-none-any.whl.metadata


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

# Passo 3: Entender como funcionam requisições na internet.

In [4]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

url = 'https://www.fundamentus.com.br/resultado.php'

driver.get(url)

# Passo 4: Entender como sites funcionam.

# Passo 5: Mapear o processo de coleta de dados no site do Fundamentus.

https://www.fundamentus.com.br/index.php

# Passo 6 - Ler a tabela de dados.

In [5]:
local_tabela = '/html/body/div[1]/div[2]/table'

elemento = driver.find_element("xpath", local_tabela)

html_tabela = elemento.get_attribute('outerHTML')

tabela = pd.read_html(str(html_tabela), thousands = '.', decimal = ',')[0]

# Passo 7 - Construir a tabela final.

In [6]:
tabela = tabela.set_index("Papel")
tabela = tabela[['Cotação', 'EV/EBIT', 'ROIC', 'Liq.2meses']]

tabela

,Cotação,EV/EBIT,ROIC,Liq.2meses
Papel,,,,
CSTB4,147.69,0.00,"22,40%",0.0
MNSA4,0.47,0.00,"-13,50%",0.0
PORP4,2.40,0.00,"0,00%",0.0
POPR4,10.17,0.00,"15,25%",0.0
IVTT3,0.00,0.00,"0,00%",0.0
...,...,...,...,...
PRBC4,14.54,0.00,"0,00%",0.0
UBBR4,7.49,0.00,"0,00%",0.0
UBBR11,14.75,0.00,"0,00%",0.0


### Temos que transformar o texto em números..

In [7]:
tabela.info()

#tabela['EV/EBIT'] * tabela['ROIC']

<class 'pandas.core.frame.DataFrame'>
Index: 987 entries, CSTB4 to SQIA3
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Cotação     987 non-null    float64
 1   EV/EBIT     987 non-null    float64
 2   ROIC        987 non-null    object 
 3   Liq.2meses  987 non-null    float64
dtypes: float64(3), object(1)
memory usage: 38.6+ KB


# Passo 7.2: Tratar o texto.

In [8]:
tabela['ROIC'] = tabela['ROIC'].str.replace("%", "")
tabela['ROIC'] = tabela['ROIC'].str.replace(".", "")
tabela['ROIC'] = tabela['ROIC'].str.replace(",", ".")
tabela['ROIC'] = tabela['ROIC'].astype(float)
display(tabela)

C:\Users\Carlos\AppData\Local\Temp\ipykernel_12432\2454759645.py:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  tabela['ROIC'] = tabela['ROIC'].str.replace(".", "")


,Cotação,EV/EBIT,ROIC,Liq.2meses
Papel,,,,
CSTB4,147.69,0.00,22.40,0.0
MNSA4,0.47,0.00,-13.50,0.0
PORP4,2.40,0.00,0.00,0.0
POPR4,10.17,0.00,15.25,0.0
IVTT3,0.00,0.00,0.00,0.0
...,...,...,...,...
PRBC4,14.54,0.00,0.00,0.0
UBBR4,7.49,0.00,0.00,0.0
UBBR11,14.75,0.00,0.00,0.0


# Passo 7.3: Fazendo os filtros e criando os rankings. 

In [9]:
tabela = tabela[tabela['Liq.2meses'] > 1000000]

In [10]:
tabela = tabela[tabela['EV/EBIT'] > 0]
tabela = tabela[tabela['ROIC'] > 0]

In [11]:
tabela['ranking_ev_ebit'] = tabela['EV/EBIT'].rank(ascending = True)
tabela['ranking_roic'] = tabela['ROIC'].rank(ascending = False)
tabela['ranking_total'] = tabela['ranking_ev_ebit'] + tabela['ranking_roic']

tabela = tabela.sort_values('ranking_total')

tabela.head(10)

,Cotação,EV/EBIT,ROIC,Liq.2meses,ranking_ev_ebit,ranking_roic,ranking_total
Papel,,,,,,,
PSSA3,27.52,0.26,60.76,5.775590e+07,1.0,1.0,2.0
PETR4,31.52,2.41,28.95,1.602730e+09,4.0,10.5,14.5
PETR3,34.49,2.56,28.95,4.355100e+08,5.0,10.5,15.5
UNIP3,71.77,4.51,38.68,1.011170e+06,14.5,3.5,18.0
WIZC3,5.91,2.20,24.18,4.456770e+06,3.0,17.0,20.0
KEPL3,10.63,4.71,43.61,1.530740e+07,20.0,2.0,22.0
FESA4,43.08,4.06,25.57,9.651050e+06,8.0,15.0,23.0
UNIP6,76.40,4.81,38.68,1.710480e+07,21.0,3.5,24.5
CMIN3,4.06,4.57,28.49,2.922810e+07,18.0,12.0,30.0
